# 07 — Makinat me vektorë mbështetës

## Konteksti dhe problemi shkencor

Regjimet në hapësirën e fazës krijojnë një problem ku margjina dhe kerneli RBF kanë interpretim gjeometrik të qartë.

### Objektivat e të nxënit

- të kuptoni margjinën dhe support vectors;
- të eksploroni C dhe gamma;
- të përdorni kernelin RBF;
- të diagnostikoni scaling dhe overfitting.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Hapësira e fazës dhe kufiri jolinear

In [ ]:
df=pd.read_csv(DATA/'phase_space_svm.csv'); X=df[['x_phase','v_phase']].to_numpy(); y=df.regjimi.to_numpy(); plt.scatter(X[:,0],X[:,1],c=y,s=14); plt.axis('equal'); plt.show()

## 2. Margjina lineare në një dataset të thjeshtë

In [ ]:
rng=np.random.default_rng(0); Xa=np.r_[rng.normal([-1,-1],.35,(80,2)),rng.normal([1,1],.35,(80,2))]; ya=np.r_[np.zeros(80),np.ones(80)]; from sklearn.svm import SVC; lin=SVC(kernel='linear',C=10).fit(Xa,ya); print('support vectors=',lin.support_vectors_.shape[0])

In [ ]:
w=lin.coef_[0]; b=lin.intercept_[0]; xx=np.linspace(-2,2,200); boundary=-(w[0]*xx+b)/w[1]; margin1=-(w[0]*xx+b-1)/w[1]; margin2=-(w[0]*xx+b+1)/w[1]; plt.scatter(Xa[:,0],Xa[:,1],c=ya,s=14); plt.plot(xx,boundary); plt.plot(xx,margin1,'--'); plt.plot(xx,margin2,'--'); plt.scatter(lin.support_vectors_[:,0],lin.support_vectors_[:,1],facecolors='none',edgecolors='k',s=90); plt.show()

## 3. Roli i C

In [ ]:
for C in [.05,.5,5,500]:
 m=SVC(kernel='linear',C=C).fit(Xa,ya); print(C,len(m.support_),m.score(Xa,ya))

## 4. RBF për regjimet rrethore

In [ ]:
for kernel in ['linear','rbf']:
 p=make_pipeline(StandardScaler(),SVC(kernel=kernel,C=3,gamma='scale')); print(kernel,cross_val_score(p,X,y,cv=5).mean())

## 5. Harta C–gamma

In [ ]:
Cs=np.logspace(-1,2,5); gs=np.logspace(-2,1,5); mat=np.zeros((len(Cs),len(gs)))
for i,C in enumerate(Cs):
 for j,g in enumerate(gs): mat[i,j]=cross_val_score(make_pipeline(StandardScaler(),SVC(C=C,gamma=g)),X,y,cv=4).mean()
plt.imshow(mat,origin='lower',aspect='auto'); plt.xticks(range(len(gs)),[f'{g:.2g}' for g in gs]); plt.yticks(range(len(Cs)),[f'{c:.2g}' for c in Cs]); plt.xlabel('gamma'); plt.ylabel('C'); plt.colorbar(label='CV accuracy'); plt.show()

## 6. Vizualizimi i kufirit RBF

In [ ]:
p=make_pipeline(StandardScaler(),SVC(C=3,gamma='scale')).fit(X,y); xx,yy=np.meshgrid(np.linspace(-1.8,1.8,250),np.linspace(-1.8,1.8,250)); zz=p.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape); plt.contourf(xx,yy,zz,alpha=.25); plt.scatter(X[:,0],X[:,1],c=y,s=12); plt.show()

## 7. Rast dështimi: scaling

In [ ]:
Xbad=X.copy(); Xbad[:,0]*=10000; print('no scaling',cross_val_score(SVC(),Xbad,y,cv=5).mean()); print('pipeline',cross_val_score(make_pipeline(StandardScaler(),SVC()),Xbad,y,cv=5).mean())

## 8. Interpretimi i support vectors

In [ ]:
m=make_pipeline(StandardScaler(),SVC(C=3,gamma='scale')).fit(X,y); svc=m[-1]; print('support vectors per class=',svc.n_support_)

## 9. Ushtrime

- Derivoni gjerësinë $2/||w||$.
- Studioni numrin e support vectors ndaj C.
- Krahasoni linear, polynomial dhe RBF.
- Shpjegoni pse grid search duhet të jetë brenda development data.

## 10. Shembull i plotë

Zgjidhni C dhe gamma me GridSearchCV në development set; pastaj vlerësoni një test final të paprekur.

In [ ]:
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,stratify=y,random_state=7); gs=GridSearchCV(make_pipeline(StandardScaler(),SVC()),{'svc__C':[.1,1,10,100],'svc__gamma':['scale',.1,1,10]},cv=5); gs.fit(Xdev,ydev); print(gs.best_params_,gs.best_score_,gs.score(Xtest,ytest))

## 11. Playground

Shtoni outliers afër qendrës dhe studioni si ndryshon kufiri për C të vogël dhe të madh.